In [0]:
# Import functions
from pyspark.sql.functions import col, current_timestamp

# Define variables used in code below
file_path = "/FileStore/stream/events/"
username = spark.sql("SELECT regexp_replace(current_user(), '[^a-zA-Z0-9]', '_')").first()[0]
table_name = f"{username}_etl_quickstart"
checkpoint_path = f"/tmp/{username}/_checkpoint/etl_quickstart"

# Clear out data from previous demo execution
spark.sql(f"DROP TABLE IF EXISTS {table_name}")
dbutils.fs.rm(checkpoint_path, True)



Out[1]: False

In [0]:
dbutils.fs.rm(f"dbfs:/user/hive/warehouse/{username}_etl_quickstart", True)


Out[2]: False

In [0]:
%fs ls /databricks-datasets/structured-streaming/events

path,name,size,modificationTime
dbfs:/databricks-datasets/structured-streaming/events/file-0.json,file-0.json,72530,1469673865000
dbfs:/databricks-datasets/structured-streaming/events/file-1.json,file-1.json,72961,1469673866000
dbfs:/databricks-datasets/structured-streaming/events/file-10.json,file-10.json,73025,1469673878000
dbfs:/databricks-datasets/structured-streaming/events/file-11.json,file-11.json,72999,1469673879000
dbfs:/databricks-datasets/structured-streaming/events/file-12.json,file-12.json,72987,1469673880000
dbfs:/databricks-datasets/structured-streaming/events/file-13.json,file-13.json,73006,1469673881000
dbfs:/databricks-datasets/structured-streaming/events/file-14.json,file-14.json,73003,1469673882000
dbfs:/databricks-datasets/structured-streaming/events/file-15.json,file-15.json,73007,1469673883000
dbfs:/databricks-datasets/structured-streaming/events/file-16.json,file-16.json,72978,1469673885000
dbfs:/databricks-datasets/structured-streaming/events/file-17.json,file-17.json,73008,1469673886000


Żeby sprawdzić czy stream działa kopjuj po jednym bądź kilku plikach 

In [0]:
dbutils.fs.cp("/databricks-datasets/structured-streaming/events/file-4.json","/FileStore/stream/events/file-4.json",True)

Out[3]: True

In [0]:
display(dbutils.fs.ls("/FileStore/stream/events/"))

path,name,size,modificationTime
dbfs:/FileStore/stream/events/file-4.json,file-4.json,72992,1745948716000


In [0]:
spark.read.format("json").load("dbfs:/FileStore/stream/events/").count()

Out[5]: 2000

Dokończyć kod autoloadera 
1. Dodaj opcje 'cloudfiles'
2. Dodaj kolumnę z metadanych 'source_file'
3. Dane zapisać do tabeli

In [0]:
%python
from pyspark.sql.functions import input_file_name

(
  spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json") 
    .option("cloudFiles.includeExistingFiles", "true")
    .load(file_path)
    .withColumn("source_file", input_file_name())
    .writeStream
    .option("cp", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)

---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-2664435758705084>:4
      1 from pyspark.sql.functions import input_file_name
      3 (
----> 4   spark.readStream
      5     .format("cloudFiles")
      6     .option("cloudFiles.format", "json") 
      7     .option("cloudFiles.includeExistingFiles", "true")
      8     .load(file_path)
      9     .withColumn("source_file", input_file_name())
     10     .writeStream
     11     .option("cp", checkpoint_path)
     12     .trigger(availableNow=True)
     13     .toTable(table_name)
     14 )

File /databricks/spark/python/pyspark/sql/streaming/readwriter.py:275, in DataStreamReader.load(self, path, format, schema, **options)
    270     if type(path) != str or len(path.strip()) == 0:
    271         raise ValueError(
    272             "If the path is provided for stream, it needs to be a "
    273             + "non-e

In [0]:
df = spark.sql(f"select count(*) from {table_name}")
df.display()

Sprawdzć metadane 

In [0]:
display(dbutils.fs.ls(f"/tmp/{username}/_checkpoint/etl_quickstart/sources/0/metadata/"))

In [0]:
%fs ls /tmp/{user}/_checkpoint/etl_quickstart/metadata